[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LinkedInLearning/Fine-Tuning-for-LLMs-from-Beginner-to-Advanced/blob/main/CH04/04_06_end/notebooks/04_06_build_training_dataset.ipynb)

# 04_06 Challenge: Build the Ridgeline training dataset

Build the complete dataset pipeline in one function. The last code cell checks your work.

These steps don't need a GPU. The default CPU runtime works fine.

In [ ]:
# Pull the course files into this Colab session and load the Ridgeline helpers.
!git clone --quiet https://github.com/LinkedInLearning/Fine-Tuning-for-LLMs-from-Beginner-to-Advanced.git course 2>/dev/null || echo "Course files already downloaded."

import sys
sys.path.insert(0, "course/CH04")

import ridgeline

In [ ]:
raw = ridgeline.load_tickets(ridgeline.DATA / "synthetic_raw.jsonl")
seeds = ridgeline.load_tickets()
print(f"{len(raw)} synthetic tickets, {len(seeds)} hand-written seed tickets")

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("unsloth/Qwen3-4B-Instruct-2507")

## Your task

Write `build_dataset` so it turns the raw tickets into clean training and evaluation sets:

1. Drop any ticket with a missing field, an unknown category, an escalate value other than Yes or No, a reply that breaks a voice rule, or possible personal information.
2. Drop tickets whose message duplicates another ticket's message.
3. Convert each ticket with `ridgeline.to_training_example` and drop any example over `max_tokens`. Count tokens with `ridgeline.count_tokens(tokenizer, example["messages"])`.
4. Put all 18 seed tickets, plus `eval_per_category` synthetic examples from each category, in the evaluation set.
5. Cap every category at `cap_per_category` training examples, so no category dominates.
6. Return the training set, the evaluation set, and a report of how many tickets you dropped and why.

In [ ]:
import random
import re
from collections import Counter

REQUIRED = ["customer_name", "subject", "message", "context", "category", "escalate", "reply"]
PII_PATTERNS = {
    "email address": r"[\w.+-]+@[\w-]+\.[\w.]+",
    "phone number": r"\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b",
    "card number": r"\b(?:\d[ -]?){13,16}\b",
}


def find_problems(ticket):
    missing = [field for field in REQUIRED if not ticket.get(field)]
    if missing:
        return ["missing field"]
    problems = []
    if ticket["category"] not in ridgeline.CATEGORIES:
        problems.append("unknown category")
    if ticket["escalate"] not in ("Yes", "No"):
        problems.append("bad escalate value")
    if ridgeline.check_reply(ticket, ridgeline.gold_output(ticket))["failures"]:
        problems.append("voice rule")
    text = " ".join(ticket[field] for field in REQUIRED)
    if any(re.search(pattern, text) for pattern in PII_PATTERNS.values()):
        problems.append("personal information")
    return problems


def build_dataset(raw, seeds, tokenizer, cap_per_category=75, eval_per_category=5, max_tokens=2048):
    report = Counter()
    rng = random.Random(42)

    # 1. Validate
    clean = []
    for ticket in raw:
        problems = find_problems(ticket)
        report.update(problems)
        if not problems:
            clean.append(ticket)

    # 2. Remove duplicates
    seen, unique = set(), []
    for ticket in clean:
        key = " ".join(ticket["message"].lower().split())
        if key in seen:
            report["duplicate"] += 1
        else:
            seen.add(key)
            unique.append(ticket)

    # 3. Format and check length
    examples = []
    for ticket in unique:
        example = ridgeline.to_training_example(ticket)
        if ridgeline.count_tokens(tokenizer, example["messages"]) > max_tokens:
            report["too long"] += 1
        else:
            examples.append(example)

    # 4 and 5. Split, then cap each category
    eval_set = [ridgeline.to_training_example(t) for t in seeds]
    train_set = []
    for category in ridgeline.CATEGORIES:
        group = [ex for ex in examples if ex["category"] == category]
        rng.shuffle(group)
        eval_set += group[:eval_per_category]
        rest = group[eval_per_category:]
        if len(rest) > cap_per_category:
            report["over category cap"] += len(rest) - cap_per_category
        train_set += rest[:cap_per_category]
    rng.shuffle(train_set)

    return train_set, eval_set, report

## Run your pipeline

In [ ]:
train_set, eval_set, report = build_dataset(raw, seeds, tokenizer)

print(f"Training examples: {len(train_set)}")
print(f"Evaluation examples: {len(eval_set)}")
print("\nDropped:")
for reason, count in sorted(report.items(), key=lambda item: -item[1]):
    print(f"  {count:>3}  {reason}")

## Check your work

In [ ]:
# Check your work. Each line prints PASS or FAIL.
from collections import Counter

def check(label, passed):
    print(f"{'PASS' if passed else 'FAIL'}  {label}")

train_ids = {ex["id"] for ex in train_set}
eval_ids = {ex["id"] for ex in eval_set}
check("Training set is not empty", len(train_set) > 0)
check("All 18 seed tickets are in the evaluation set", {t["id"] for t in seeds} <= eval_ids)
check("No example appears in both sets", not (train_ids & eval_ids))
check("No category has more than 75 training examples",
      bool(train_set) and max(Counter(ex["category"] for ex in train_set).values()) <= 75)
check("Every training reply passes the voice rules", bool(train_set) and all(
    ridgeline.check_reply({"id": ex["id"], "customer_name": ex["messages"][1]["content"].split("\n")[0].removeprefix("Customer: ")},
                          ex["messages"][-1]["content"])["voice_ok"] for ex in train_set))
check("Every example has system, user, and assistant messages",
      all([m["role"] for m in ex["messages"]] == ["system", "user", "assistant"] for ex in train_set + eval_set) and bool(train_set))

## Save your dataset

In [ ]:
ridgeline.save_jsonl(train_set, "train.jsonl")
ridgeline.save_jsonl(eval_set, "eval.jsonl")
print("Saved train.jsonl and eval.jsonl. Download them from the Files panel to use your own data in the next chapter.")